In [37]:
import pandas as pd
import re

In [38]:
# ===============================
# LOAD CSV
# ===============================
CSV_PATH = "ugf27.csv"   # change if needed
df = pd.read_csv(CSV_PATH)
cols = list(df.columns)

In [39]:
# ===============================
# HELPERS
# ===============================
def clean_number(x: str) -> float:
    """
    Removes pandas-added suffixes like .1 .12 .128
    """
    x = re.sub(r"\.\d+$", "", x)
    return float(x)


def extract_abc_from_left(cols, start_idx):
    """
    Scan LEFT from ugf column to find nearest a, b, c headers
    Works for first block (no suffix) and later blocks
    """
    a = b = c = None

    for i in range(start_idx, -1, -1):
        if a is None:
            m = re.search(r"a=([0-9.eE+-]+)", cols[i])
            if m:
                a = clean_number(m.group(1))

        if b is None:
            m = re.search(r"b=([0-9.eE+-]+)", cols[i])
            if m:
                b = clean_number(m.group(1))

        if c is None:
            m = re.search(r"c=([0-9.eE+-]+)", cols[i])
            if m:
                c = clean_number(m.group(1))

        if a is not None and b is not None and c is not None:
            return a, b, c

    return None


In [40]:
# ===============================
# CORE EXTRACTION
# ===============================
rows = []
block_id = 0

for ugf_idx in range(1, len(cols), 2):   # odd indices = ugf columns
    ugf_col = cols[ugf_idx]
    d_col = cols[ugf_idx - 1]            # paired d column (CRITICAL FIX)

    abc = extract_abc_from_left(cols, ugf_idx)
    if abc is None:
        continue

    a, b, c = abc

    d_vals = pd.to_numeric(df[d_col], errors="coerce")
    ugf_vals = pd.to_numeric(df[ugf_col], errors="coerce")

    valid_rows = ugf_vals.notna().sum()
    if valid_rows == 0:
        print(f"⛔ Block {block_id}: empty ugf column → skipped")
        continue

    # ================= DEBUG =================
    print("\n-----------------------------")
    print(f"BLOCK {block_id}")
    print(f"a = {a}, b = {b}, c = {c}")
    print(f"d column   : {d_col}")
    print(f"ugf column: {ugf_col}")
    print(f"valid ugf rows: {valid_rows}")
    # =========================================

    added = 0
    for d, ugf in zip(d_vals, ugf_vals):
        if pd.isna(d) or pd.isna(ugf):
            continue

        rows.append({
            "a": a,
            "b": b,
            "c": c,
            "d": d,
            "ugf": ugf
        })
        added += 1

    print(f"✅ Rows added from block: {added}")
    block_id += 1


# ===============================
# FINAL DATAFRAME
# ===============================
final_df = pd.DataFrame(rows)

print("\n===============================")
print("FINAL SHAPE:", final_df.shape)
print(final_df.head())


# ===============================
# HARD VALIDATION
# ===============================
violations = (
    final_df
    .groupby(["a", "b", "c", "d"])["ugf"]
    .nunique()
    .reset_index()
    .query("ugf > 1")
)

print("\n===============================")
if len(violations) > 0:
    print("❌ ERROR: Multiple ugfs for same (a,b,c,d)")
    print(violations.head())
else:
    print("✅ PASSED: Exactly one ugf per (a,b,c,d)")


-----------------------------
BLOCK 0
a = 1.12e-05, b = 0.0, c = 2.4e-06
d column   : c=2.4e-06
ugf column: a=1.12e-05.1
valid ugf rows: 13
✅ Rows added from block: 13

-----------------------------
BLOCK 1
a = 1.12e-05, b = 0.000132012, c = 2.4e-06
d column   : b=0.000132012.1
ugf column: c=2.4e-06.1
valid ugf rows: 13
✅ Rows added from block: 13

-----------------------------
BLOCK 2
a = 1.12e-05, b = 0.000132012, c = 2.4e-06
d column   : a=1.12e-05.2
ugf column: b=0.000132012.2
valid ugf rows: 13
✅ Rows added from block: 13

-----------------------------
BLOCK 3
a = 1.12e-05, b = 0.000132012, c = 2.666667e-06
d column   : c=2.666667e-06
ugf column: a=1.12e-05.3
valid ugf rows: 13
✅ Rows added from block: 13

-----------------------------
BLOCK 4
a = 1.12e-05, b = 0.000132012, c = 2.666667e-06
d column   : b=0.000132012.3
ugf column: c=2.666667e-06.1
valid ugf rows: 13
✅ Rows added from block: 13

-----------------------------
BLOCK 5
a = 1.12e-05, b = 0.000132012, c = 2.666667e-06


In [41]:
# ===============================
# DUPLICATE ROW CHECK
# ===============================
dups = final_df.duplicated(subset=["a", "b", "c", "d"], keep=False)

print("\n===============================")
print("Duplicate (a,b,c,d) rows:", dups.sum())

if dups.any():
    print(final_df.loc[dups].head())
else:
    print("✅ No duplicate rows found")


Duplicate (a,b,c,d) rows: 6565
                  a                 b                 c                 d  \
0 0.000011200000000 0.000000000000000 0.000002400000000 0.000038400000000   
1 0.000011200000000 0.000000000000000 0.000002400000000 0.000042700000000   
2 0.000011200000000 0.000000000000000 0.000002400000000 0.000046900000000   
3 0.000011200000000 0.000000000000000 0.000002400000000 0.000051200000000   
4 0.000011200000000 0.000000000000000 0.000002400000000 0.000055500000000   

                       ugf  
0 12930777.429999999701977  
1 14343496.060000000521541  
2 15536351.640000000596046  
3 17481919.760000001639128  
4 19550530.789999999105930  


In [42]:
before = len(final_df)

final_df = final_df.drop_duplicates(
    subset=["a", "b", "c", "d"],
    keep="first"   # keep original occurrence
)

after = len(final_df)

print(f"Dropped {before - after} duplicate rows")
print("Final shape after dedup:", final_df.shape)

Dropped 4355 duplicate rows
Final shape after dedup: (2223, 5)


In [43]:
dup_cnt = final_df.duplicated(subset=["a", "b", "c", "d"]).sum()
print("Remaining duplicate (a,b,c,d):", dup_cnt)


Remaining duplicate (a,b,c,d): 0


In [44]:
pd.set_option("display.float_format", "{:.15f}".format)

In [45]:
final_df.shape

(2223, 5)

In [46]:
final_df.head(39)

,a,b,c,d,ugf
0,0.000011200000000,0.000000000000000,0.000002400000000,0.000038400000000,12930777.429999999701977
1,0.000011200000000,0.000000000000000,0.000002400000000,0.000042700000000,14343496.060000000521541
2,0.000011200000000,0.000000000000000,0.000002400000000,0.000046900000000,15536351.640000000596046
3,0.000011200000000,0.000000000000000,0.000002400000000,0.000051200000000,17481919.760000001639128
4,0.000011200000000,0.000000000000000,0.000002400000000,0.000055500000000,19550530.789999999105930
5,0.000011200000000,0.000000000000000,0.000002400000000,0.000059700000000,21397590.879999998956919
6,0.000011200000000,0.000000000000000,0.000002400000000,0.000064000000000,23058497.920000001788139
7,0.000011200000000,0.000000000000000,0.000002400000000,0.000068300000000,25321909.410000000149012
8,0.000011200000000,0.000000000000000,0.000002400000000,0.000072500000000,28497489.760000001639128
9,0.000011200000000,0.000000000000000,0.000002400000000,0.000076800000000,31420489.920000001788139


In [47]:
ml_ready = final_df[13:]

In [48]:
ml_ready.shape

(2210, 5)

In [49]:
ml_ready.to_csv('ugf_ml.csv', index=False)